In [213]:
import pandas as pd
import numpy as np

In [214]:
bmp_ump = pd. read_excel("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/display files/BMP_UMP.xlsx")

In [215]:
kam_nkam = pd.read_excel("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/display files/KAM_NKAM.xlsx")

In [216]:
pipeline = pd.read_excel("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/12.03/Display/Pipeline Report.xlsx",sheet_name="Line Item Export")

In [217]:
billing = pd.read_excel("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/12.03/Display/Billing Report.xlsx",sheet_name="LineItem Report")

In [218]:
search_billing = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/12.03/Display/Search Billing Sign off Mar'26 - Accruals - Final Billing Sign off.csv",skiprows=1,skipfooter=9,engine='python')

In [219]:
advance = pd.read_csv("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/12.03/Display/Advance Report.csv",skiprows = 2)

In [220]:
master_classification = pd.read_excel("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/display files/Master_classification.xlsx")

In [221]:
Tagging = pd.read_excel("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/display files/Tagging.xlsx")

In [222]:
supercat_wise =pd.read_excel("/content/drive/MyDrive/Ads Biz-fin/Estimate Working/CY'26/3.Mar'26/display files/Supercat wise.xlsx")

In [223]:
# supercat_wise.columns

In [224]:
kam_nkam = kam_nkam.rename(columns={"Concatenate":"BUxBrand"})

- Function to Convert Object to INT64

In [225]:
def obj2int(series):
    return (
        series.astype(str)
              .str.replace(',', '', regex=False)
              .astype(float)
              .round(0)
              .astype('Int64')
    )

In [226]:
def flt2int(series):

    if series.dtype == 'object':
        series = series.str.replace(',', '', regex=False)

    return (
        pd.to_numeric(series, errors='coerce') # Invalid strings become NaN
        .round(0)
        .astype('Int64')
    )


# Pipeline Report


In [227]:
 pipeline.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2658 entries, 0 to 2657
Data columns (total 43 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   RO Number                          2657 non-null   object        
 1   RO Brand                           2657 non-null   object        
 2   RO Brand Text                      218 non-null    object        
 3   RO End Date                        2657 non-null   datetime64[ns]
 4   Unallocated in RO                  2658 non-null   float64       
 5   Super Category                     2657 non-null   object        
 6   Advertiser                         2658 non-null   object        
 7   Brand                              2658 non-null   object        
 8   Lead Status                        2658 non-null   object        
 9   Lead Id                            2658 non-null   int64         
 10  Lead Name                          2

In [228]:
# advance.info()

In [229]:
# billing.info()

- Clean pipeline Data


In [230]:
pipeline["SuperCatagory"] = (
    pipeline["Super Category"]
    .str.replace("_deleted", "", regex=False)
    .str.replace("deleted", "", regex=False)
    .str.strip()
)

In [231]:
pipeline['Line Item Net Budget'].sum()

np.float64(2271784429.58)

- Date Check

In [232]:
StartOfMonth = '2026-03-01'
pipeline['Date Check'] = pipeline['Line Item Start Date'] < StartOfMonth

pipeline['Date Check'].value_counts()

,count
Date Check,
False,2244
True,414


- Days

In [233]:
Pipeline_Date = pd.to_datetime('2026-03-11')

In [234]:
diff1 = (Pipeline_Date - pipeline['Line Item Start Date']).dt.days
diff2 = (pipeline['Line Item End Date'] - pipeline['Line Item Start Date']).dt.days

#It ensures that any negative results are reset to zero.

pipeline['Days'] = np.minimum(diff1, diff2).clip(0)


In [235]:
pipeline['Days'].sum()

np.int64(22272)

- Consumption

In [236]:
duration = (pipeline['Line Item End Date'] - pipeline['Line Item Start Date']).dt.days.clip(lower=1)

pipeline['Consumption'] = np.where(
    pipeline['Date Check'].astype(str) == "True",
    0,
    (pipeline['Line Item Net Budget'] / duration) * pipeline['Days']
)


In [237]:
pipeline['Consumption'].sum()

np.float64(821693187.1620066)

In [238]:
Exception_map = [157770,157771,157773,157774]
pipeline['Exception'] = np.where(pipeline['Line Item Id'].isin(Exception_map), 'Exception', '0')

- Exception

In [239]:
pipeline['Exception'].value_counts()

,count
Exception,
0,2658


- Estimates

In [240]:
pipeline['Final_Estimated'] = np.where(pipeline['Exception'] == "Exception", 0, np.where(pipeline['Date Check'] == True, 0, pipeline['Line Item Net Budget']))

In [241]:
pipeline['Final_Estimated'].sum()

np.float64(1703622604.1399999)

- Rate Card Bifurcation

In [242]:
rateCard_map = dict(zip(Tagging['Rate Card'],Tagging['Type_RC']))
RC_map = dict(zip(Tagging['RC_type_Tag'],Tagging['RC_Tag']))
pipeline['Rate Card Bifurcation'] = pipeline['Rate Card Dimension'].map(rateCard_map).fillna(pipeline['Rate Card Dimension'].map(RC_map)).fillna(0)

In [243]:
# pipeline['Rate Card Bifurcation'].value_counts()

 - Alpha/MP

In [244]:
pipeline['Alpha_MP_Tag'] = np.where(pipeline['SuperCatagory']=="Quich","HL",np.where(pipeline['Line Item Business Type']=="",pipeline['RO Business Type'],pipeline['Line Item Business Type']))

In [245]:
# pipeline['Alpha_MP_Tag'].value_counts()

In [246]:
pipeline['New Alpha/MP'] = pipeline['Alpha_MP_Tag'].replace({"BMP": "MP", "PPA - MP": "3P", "PPA - Alpha": "3P"})


In [247]:
# pipeline['New Alpha/MP'].value_counts()

- New Supercategory

In [248]:

map_sheet1 = Tagging.drop_duplicates('Brand Name').set_index('Brand Name')['HL Allocation']

map_mc_b = master_classification.drop_duplicates('Category').set_index('Category')['Supercategory']
map_mc_c = master_classification.drop_duplicates('BUSupercat').set_index('BUSupercat')['Supercategory']

temp_df = master_classification.drop_duplicates('Supercategory').set_index('Supercategory')
map_mc_e = temp_df.index.to_series()

conditions = [
    (pipeline['SuperCatagory'] == "Quick DBEFM"),(pipeline['SuperCatagory'] == "Quick Grocery"),
    (pipeline['SuperCatagory'] == "Quick Staples"),(pipeline['SuperCatagory'] == "Quick Healthcare"),
    (pipeline['SuperCatagory'] == "Quick Home"),(pipeline['SuperCatagory'] == "Quick Foods"),
    (pipeline['SuperCatagory'].str.contains("Quick", na=False))
             ]

choices = [ "FoodAndNutrition", "Grocery", "Staples", "Healthcare", "Home", "FoodAndNutrition",
              pipeline['RO Brand'].map(map_sheet1).fillna(0)
          ]

default_lookup = pipeline['SuperCatagory'].map(map_mc_b) \
                 .fillna(pipeline['SuperCatagory'].map(map_mc_c)) \
                 .fillna(pipeline['SuperCatagory'].map(map_mc_e)) \
                 .fillna(0)

pipeline['New Supercat'] = np.select(conditions, choices, default=default_lookup)


- New BU

In [249]:
bu_mapping = dict(zip(master_classification['Supercategory'],master_classification['BU']))
pipeline['New BU'] = pipeline['New Supercat'].map(bu_mapping).fillna(
    pipeline['New Supercat'].replace({"Staples": "Grocery", "Powerbank": "Electronics"})
)

In [250]:
# pipeline['New BU'].value_counts()

- RO Start Date

In [251]:

pipeline = pipeline.reset_index(drop=True)

date_mapping = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['RO Start Date']

pipeline['RO Start Date'] = pipeline['Line Item Id'].map(date_mapping).fillna(pd.to_datetime('1900-01-01'))


- Check

In [252]:
last_month_end = '2026-02-28'
pipeline['Check'] = np.where(pipeline['RO End Date'] >= pipeline['RO Start Date'] ,"Yes","No")

In [253]:
# pipeline['Check'].value_counts()

- RO Amount

In [254]:

sumifs_q = pipeline.groupby('RO Number')['Line Item Net Budget'].transform('sum')

billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['RO Amount']
xlookup_val = pipeline['Line Item Id'].map(billing_map).fillna(0)

SOM_date = pd.to_datetime('2026-03-01')

condition = (pipeline['Check'] == "Yes") & (pipeline['RO Start Date'] >= SOM_date)

calculation = np.divide(xlookup_val, sumifs_q, out=np.zeros_like(xlookup_val), where=sumifs_q!=0) * pipeline['Line Item Net Budget']


pipeline['RO Amount'] = np.where(condition, calculation, 0)


In [255]:
pipeline['RO Amount'].sum()

np.float64(1705016366.9984639)

In [256]:
pipeline['SCxBrandxAlpha_MP'] = pipeline['New Supercat'].astype(str) + pipeline['RO Brand'].astype(str) + pipeline['New Alpha/MP'].astype(str)
PB_mapping = Tagging.drop_duplicates('SCxBrandxType').set_index('SCxBrandxType')['PrivateBrand']
pipeline['PrivateBrand'] = pipeline['SCxBrandxAlpha_MP'].map(PB_mapping).fillna("0")

In [257]:
# pipeline['PrivateBrand'].value_counts()

- BMP/UMP

In [258]:
def BMP_NonBMP(df):
  if df['New Supercat'] == "Automobile":
    return "BMP"
  elif df['Line Item Business Type'] == "BMP":
    return "BMP"
  else:
    return "Non BMP"

In [259]:
pipeline['BMP_UMP'] = pipeline.apply(BMP_NonBMP, axis=1)

In [260]:
pipeline['BMP_UMP'].value_counts()

,count
BMP_UMP,
Non BMP,2518
BMP,140


- KAM/NKAM

In [261]:
pipeline['BUxBrand'] = pipeline['New BU'].astype(str).str.upper().str.strip() + pipeline['RO Brand'].astype(str).str.upper().str.strip()

In [262]:
kam_nkam['BUxBrand'] = kam_nkam['BUxBrand'].astype(str).str.upper().str.strip()
kam_nkam['Brand-1'] = kam_nkam['Brand-1'].astype(str).str.upper().str.strip()
kam_nkam['Owner'] = kam_nkam['Owner'].astype(str).str.upper().str.strip()
pipeline['BUxBrand'] = pipeline['BUxBrand'].astype(str).str.upper().str.strip()
pipeline['Brand'] = pipeline['Brand'].astype(str).str.upper().str.strip()

mapping_BUXBrand = kam_nkam.set_index('BUxBrand')['Owner'].to_dict()
mapping_Brand = kam_nkam.set_index('Brand-1')['Owner'].to_dict()

BUxBrand_KN = pipeline['BUxBrand'].map(mapping_BUXBrand)
Brand_KN = pipeline['Brand'].map(mapping_Brand)

conditions = [ (pipeline['New Supercat'] == "Automobile"), BUxBrand_KN.notna(), Brand_KN.notna() ]

choices = [ "KAM", BUxBrand_KN, Brand_KN ]

pipeline['kam_nkam'] = np.select(conditions, choices, default="KAM")


In [263]:
pipeline['kam_nkam'].value_counts()

,count
kam_nkam,
KAM,2045
NKAM,613


In [264]:
pipeline[pipeline['kam_nkam'] == "KAM"]['Final_Estimated'].sum()

np.float64(1143630031.26)

# Search Billing Sign Off


- Convert object to integer

In [265]:
search_billing['Net Billing Number'] = obj2int(search_billing['Net Billing Number'])
search_billing['LineItem Gross Budget'] = obj2int(search_billing['LineItem Gross Budget'])
search_billing['Net Billing Number'] = obj2int(search_billing['Net Billing Number'])
search_billing['Gross ad spend'] = obj2int(search_billing['Gross ad spend'])
search_billing['LineItem Net Budget'] = obj2int(search_billing['LineItem Net Budget'])

- Clean supercategory

In [266]:
search_billing["SuperCatagory"] = (
    search_billing["Industry.1"]
    .str.replace("_deleted", "", regex=False)
    .str.replace("deleted", "", regex=False)
    .str.strip()
)

- New Super Category

In [267]:

map_sheet1 = Tagging.drop_duplicates('Brand Name').set_index('Brand Name')['HL Allocation']

map_mc_b = master_classification.drop_duplicates('Category').set_index('Category')['Supercategory']
map_mc_c = master_classification.drop_duplicates('BUSupercat').set_index('BUSupercat')['Supercategory']

temp_df = master_classification.drop_duplicates('Supercategory').set_index('Supercategory')
map_mc_e = temp_df.index.to_series()

conditions = [
    (search_billing['SuperCatagory'] == "Quick DBEFM"),(search_billing['SuperCatagory'] == "Quick Grocery"),
    (search_billing['SuperCatagory'] == "Quick Staples"),(search_billing['SuperCatagory'] == "Quick Healthcare"),
    (search_billing['SuperCatagory'] == "Quick Home"),(search_billing['SuperCatagory'] == "Quick Foods"),
    (search_billing['SuperCatagory'].str.contains("Quick", na=False))
]

choices = [ "FoodAndNutrition", "Grocery", "Staples", "Healthcare", "Home", "FoodAndNutrition",
    search_billing['RO Brand'].map(map_sheet1).fillna(0)
]

default_lookup = search_billing['SuperCatagory'].map(map_mc_b) \
                 .fillna(search_billing['SuperCatagory'].map(map_mc_c)) \
                 .fillna(search_billing['SuperCatagory'].map(map_mc_e)) \
                 .fillna(0)

search_billing['New Supercat'] = np.select(conditions, choices, default=default_lookup)


- New BU

In [268]:
bu_mapping = dict(zip(master_classification['Supercategory'],master_classification['BU']))
search_billing['New BU'] = search_billing['New Supercat'].map(bu_mapping).fillna(
    search_billing['New Supercat'].replace({"Staples": "Grocery", "Powerbank": "Electronics"})
)

In [269]:
# search_billing['New Supercat'].value_counts()

- BMP/UMP


In [270]:
def BMP_NonBMP(df):
  if df['New Supercat'] == "Automobile":
    return "BMP"
  elif df['Line Item Business Type'] == "BMP":
    return "BMP"
  else:
    return "Non BMP"

In [271]:
search_billing['BMP_UMP'] = search_billing.apply(BMP_NonBMP, axis=1)

In [272]:
# search_billing['BMP_UMP'].value_counts()

In [273]:
search_billing["BUxBrand_Cleaned"] = search_billing["New BU"].astype(str).str.upper().str.strip() + search_billing["RO Brand"].astype(str).str.upper().str.strip()

In [274]:
BUxBrand_map =dict(zip(kam_nkam['BUxBrand'].astype(str).str.upper().str.strip(),kam_nkam['Owner'].astype(str).str.upper().str.strip()))
Brand_map = dict(zip(kam_nkam['Brand-1'].astype(str).str.upper().str.strip(),kam_nkam['Owner'].astype(str).str.upper().str.strip()))
search_billing['KAM_NKAM_check'] = search_billing['BUxBrand_Cleaned'].map(BUxBrand_map).fillna(search_billing['RO Brand'].astype(str).str.upper().str.strip().map(Brand_map)).fillna("KAM")

In [275]:
search_billing['KAM_NKAM_check'].value_counts()

,count
KAM_NKAM_check,
KAM,618
NKAM,112


In [276]:
# search_billing[search_billing["KAM_NKAM_check"]=="NKAM"]['Estimate'].sum()

- BOC/BOL


In [277]:
BOL_BOC_map = dict(zip(Tagging['Type_Dimention'],Tagging['Check']))
search_billing['BOC_BOL'] = search_billing['Billing Details'].map(BOL_BOC_map).fillna(0)

In [278]:
search_billing['BOC_BOL'].value_counts()

,count
BOC_BOL,
BOC,411
BOL,183
NO,136


- Consumption

In [279]:
search_billing["Consumption"] = np.where(search_billing['BOC_BOL']=="NO",0,
      search_billing[['LineItem Net Budget','Gross ad spend','Net Billing Number']].min(axis=1))

- Estimates

In [280]:
BQ1 = 10   # Consumed days
BR1 = 31  # Total day of current month


conditions = [
    (search_billing['BOC_BOL'] == "NO"),
    (search_billing['BOC_BOL'] == "BOL") & ((search_billing['Gross ad spend'] / search_billing['LineItem Gross Budget']).fillna(0) > 0.5)
]

choices = [ 0, search_billing['Net Billing Number'] ]

calc_weighted = (search_billing['Gross ad spend'] / BQ1) * BR1
default_case = search_billing['LineItem Net Budget'].combine(calc_weighted, min)

search_billing['Estimate'] = np.select(conditions, choices, default=default_case)


In [281]:
# search_billing.info(verbose=True,show_counts=True)

- Cons. As a % of Budget

In [282]:
search_billing["Cons. As a % of Budget"] = (search_billing['Consumption']/search_billing['Estimate']).fillna(0)

- Cons. As a % of Net Billing

In [283]:
search_billing["Cons. As a % of Net Billing"] = (search_billing['Consumption']/search_billing['Net Billing Number']).fillna(0)

- Alpha/MP

In [284]:
lookup_map = Tagging.set_index('Supercat')['Alpha/MP'].to_dict()

In [285]:
# Alpha_MP
cond_quick = search_billing['SuperCatagory'].str.contains("Quick", na=False, case=False)

cond_lookup = search_billing['New Supercat'].isin(lookup_map.keys())

choices = [
    "HL",
    search_billing['New Supercat'].map(lookup_map)
]

default_logic = np.select(
    [search_billing['Line Item Business Type'].isna() | (search_billing['Line Item Business Type'] == ""), (search_billing['Line Item Business Type'] == "BMP")],
    [search_billing['RO Business Type'], "MP"],
    default=search_billing['Line Item Business Type']
)

search_billing['Alpha_MP'] = np.select([cond_quick, cond_lookup],
                                        choices,
                                        default=default_logic)


In [286]:
search_billing['Alpha_MP'].value_counts()

,count
Alpha_MP,
MP,518
Alpha,187
HL,23
3P,2


- KAM/NKAM

In [287]:
search_billing["BUxBrand"] = search_billing["New BU"].astype(str) + search_billing["RO Brand"].astype(str)

In [288]:
mapping_BUXBrand = kam_nkam.set_index('BUxBrand')['Owner'].to_dict()
mapping_Brand = kam_nkam.set_index('Brand-1')['Owner'].to_dict()

SBUxBrand_KN = search_billing['BUxBrand'].map(mapping_BUXBrand)
SBrand_KN = search_billing['RO Brand'].map(mapping_Brand)

conditions = [ (search_billing['New Supercat'] == "Automobile"), SBUxBrand_KN.notna(), SBrand_KN.notna() ]

choices = [ "KAM", SBUxBrand_KN, SBrand_KN ]

search_billing['kam_nkam'] = np.select(conditions, choices, default="KAM")


In [289]:
search_billing['kam_nkam'].value_counts()

,count
kam_nkam,
KAM,702
NKAM,28


In [290]:
search_billing[search_billing['SuperCatagory'] == "Automobile"]['New Supercat'].value_counts()

,count
New Supercat,
Automobile,10


In [291]:
billing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2541 entries, 0 to 2540
Data columns (total 62 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   Row #                                   2537 non-null   object        
 1   Ext RO Number Flag                      2537 non-null   object        
 2   External RO number                      1162 non-null   object        
 3   Billing as per the Internal RO Number?  2536 non-null   object        
 4   Internal RO number                      2536 non-null   object        
 5   RO Business Type                        2536 non-null   object        
 6   Line Item Business Type                 2536 non-null   object        
 7   Advertiser                              2536 non-null   object        
 8   Sales Manager                           2536 non-null   object        
 9   Line Item Name                          2536 non-nul

# Advance Report


In [292]:
billing['Line Item Id'] = flt2int(billing['Line Item Id'])

In [293]:
billing['Line Item Id'].dtype

Int64Dtype()

- Line Item Id


In [294]:
advance.loc[:, 'Line Item Id'] = (
    advance["Name"].str.extract(r"(\d+)")
    .fillna(0)
    .astype(int)
)

In [295]:
advance['Line Item Id'].count()

np.int64(1504)

In [392]:
# advance[advance['Line Item Id'] == 0]

- Supercat

In [296]:
map_p1 = pipeline.set_index('Line Item Name')['SuperCatagory'].to_dict()
map_p2 = pipeline.set_index('Line Item Id')['SuperCatagory'].to_dict()
map_b3 = billing.set_index('Line Item Id')['Industry.1'].to_dict()

advance['Supercat'] = advance['Name'].map(map_p1)

advance['Supercat'] = advance['Supercat'].fillna(advance['Line Item Id'].map(map_p2))
advance['Supercat'] = advance['Supercat'].fillna(advance['Line Item Id'].map(map_b3))

advance['Supercat'] = advance['Supercat'].fillna(0)


In [297]:
# advance['Supercat'].value_counts()

- Brand


In [298]:
map_m_to_b = pipeline.drop_duplicates('Line Item Name').set_index('Line Item Name')['RO Brand']
map_l_to_b = pipeline.drop_duplicates('Line Item Id').set_index('Line Item Id')['RO Brand']

advance['Brand'] = (
    advance['Name'].map(map_m_to_b)
    .fillna(advance['Line Item Id'].map(map_l_to_b))
    .fillna(0)
)


In [299]:
# advance['Brand'].value_counts()

- New SuperCat

In [300]:

map_sheet1 = Tagging.drop_duplicates('Brand Name').set_index('Brand Name')['HL Allocation']

map_mc_ba = master_classification.drop_duplicates('Category').set_index('Category')['Supercategory']
map_mc_ca = master_classification.drop_duplicates('BUSupercat').set_index('BUSupercat')['Supercategory']

temp_df_a = master_classification.drop_duplicates('Supercategory').set_index('Supercategory')
map_mc_ea = temp_df.index.to_series()

conditions = [
    (advance['Supercat'] == "Quick DBEFM"),(advance['Supercat'] == "Quick Grocery"),
    (advance['Supercat'] == "Quick Staples"),(advance['Supercat'] == "Quick Healthcare"),
    (advance['Supercat'] == "Quick Home"),(advance['Supercat'] == "Quick Foods"),
    (advance['Supercat'].str.contains("Quick", na=False))
]

choices = [ "FoodAndNutrition", "Grocery", "Staples", "Healthcare", "Home", "FoodAndNutrition",
    advance['Brand'].map(map_sheet1).fillna(0)
]

default_lookup = advance['Supercat'].map(map_mc_b) \
                 .fillna(advance['Supercat'].map(map_mc_c)) \
                 .fillna(advance['Supercat'].map(map_mc_e)) \
                 .fillna(0)

advance['New Supercat'] = np.select(conditions, choices, default=default_lookup)


In [301]:
# advance['New Supercat'].value_counts()

- New BU


In [408]:
master_map = master_classification.drop_duplicates('Supercategory').set_index('Supercategory')['BU']

advance['New BU'] = advance['New Supercat'].map(master_map).fillna(advance['New Supercat'].replace({"Staples": "Grocery"}))


In [409]:
advance[advance['New BU']== 0]['New Supercat'].value_counts()

,count
New Supercat,
0,68


- Alpha/MP

In [410]:
map_sheet1_no = Tagging.set_index('Supercat')['Alpha/MP'].to_dict()
map_pipe_mae = pipeline.set_index('Line Item Name')['RO Business Type'].to_dict()
map_pipe_lae = pipeline.set_index('Line Item Id')['RO Business Type'].to_dict()

is_quick = advance['Supercat'].str.contains("Quick", case=False, na=False)

is_in_sheet1 = advance['New Supercat'].isin(Tagging['Supercat'])

pipeline_lookup = (
    advance['Name'].map(map_pipe_mae)
    .fillna(advance['Line Item Id'].map(map_pipe_lae))
    .fillna(0)
)

conditions = [
    is_quick,
    is_in_sheet1
]

choices = [
    "HL",
    advance['New Supercat'].map(map_sheet1_no),
]

advance['Alpha_MP'] = np.select(conditions, choices, default=pipeline_lookup)


In [411]:
advance['Alpha_MP'].value_counts()

,count
Alpha_MP,
Alpha,1008
HL,229
MP,122
0,69
BMP,43
3P,33


- New Alpha/MP

In [412]:
advance['New Alpha/MP'] = advance['Alpha_MP'].replace({"BMP": "MP", "PPA - MP": "3P", "PPA - Alpha": "3P"})


In [413]:
advance['New Alpha/MP'].value_counts()

,count
New Alpha/MP,
Alpha,1008
HL,229
MP,165
0,69
3P,33


- Line Item Net Budget

In [414]:
map_p_m_q = pipeline.drop_duplicates('Line Item Name').set_index('Line Item Name')['Line Item Net Budget']
map_p_l_q = pipeline.drop_duplicates('Line Item Id').set_index('Line Item Id')['Line Item Net Budget']
map_b_k_t = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['LineItem Net Budget']

advance['Line Item Net Budget'] = (
    advance['Name'].map(map_p_m_q)
    .fillna(advance['Line Item Id'].map(map_p_l_q))
    .fillna(advance['Line Item Id'].map(map_b_k_t))
    .fillna(0)
)


Check (Supercat Match)

In [415]:
advance['Check'] = advance['New Supercat'].isin(supercat_wise['Super Categories'])


In [309]:
# advance['Check'].value_counts()

- Start Date


In [416]:
map_m_to_o = pipeline.drop_duplicates('Line Item Name').set_index('Line Item Name')['Line Item Start Date']
map_l_to_o = pipeline.drop_duplicates('Line Item Id').set_index('Line Item Id')['Line Item Start Date']

advance['Start_Date'] = (
    advance['Name'].map(map_m_to_o)
    .fillna(advance['Line Item Id'].map(map_l_to_o))
    .fillna(pd.to_datetime('1900-01-01'))
)

In [311]:
# advance['Start_Date'].head()

- End Date

In [417]:
map_m_to_p = pipeline.drop_duplicates('Line Item Name').set_index('Line Item Name')['Line Item End Date']
map_l_to_p = pipeline.drop_duplicates('Line Item Id').set_index('Line Item Id')['Line Item End Date']

advance['End_Date'] = (
    advance['Name'].map(map_m_to_p)
    .fillna(advance['Line Item Id'].map(map_l_to_p))
    .fillna(pd.to_datetime('1900-01-01'))
)

In [313]:
# advance['End_Date'].head()

- CPD/CMP

In [418]:
map_m_to_x = pipeline.drop_duplicates('Line Item Name').set_index('Line Item Name')['Rate Card']
map_l_to_x = pipeline.drop_duplicates('Line Item Id').set_index('Line Item Id')['Rate Card']

advance['CPD_CPM'] = (
    advance['Name'].map(map_m_to_x)
    .fillna(advance['Line Item Id'].map(map_l_to_x))
    .fillna(0)
)

In [419]:
advance['CPD_CPM'].value_counts()

,count
CPD_CPM,
vCPM,1082
Less than 100 SOV,271
0,68
100 SOV - CPD,50
100 SOV,15
Associate Sponsor,7
CPC,6
LS CLP Holi Package,4
CPTS,1


- BOL/BOC

In [420]:
bol_list = ["Less than 100 SOV", "CPD", "CPT", "Reservation Buy", "100 SOV - CPD", "100 SOV" ]
advance['BOL/BOC'] = np.where(advance['CPD_CPM'].str.strip().isin(bol_list), "BOL", "BOC")

In [421]:
advance['BOL/BOC'].value_counts()

,count
BOL/BOC,
BOC,1168
BOL,336


- Date_Check


In [422]:
ap1_value = pd.to_datetime('2026-03-01')
advance['Date_Check'] = np.where(advance['BOL/BOC'] == 'BOC', "False", advance['Start_Date'] < ap1_value)

In [423]:
advance['Date_Check'].value_counts()

,count
Date_Check,
False,1494
True,10


- Date of Report

In [424]:
advance['Date of Report'] = pd.to_datetime('2026-03-11')

- No of Days


In [425]:
advance['NoofDays'] = np.maximum(
    np.minimum(
        (advance["Date of Report"] - advance["Start_Date"]).dt.days,
        (advance["End_Date"] - advance["Start_Date"]).dt.days
    ),
    1
)

In [382]:
# advance['NoofDays'].sum()

- Final Consumption Numbers

In [426]:
duration = (advance['End_Date'] - advance['Start_Date']).dt.days.clip(lower=1)

conditions = [
    (advance['Date_Check'] == "True"),
    (advance['BOL/BOC'] == "BOL")
]

choices = [
    0,
    (advance['Line Item Net Budget'] / duration) * advance['NoofDays'],
]

advance['Final Consumption Numbers'] = np.select(
    conditions,
    choices,
    default=np.minimum(advance['Net Spend'], advance['Line Item Net Budget'])
)


In [427]:
advance['Final Consumption Numbers'].sum()

np.float64(807330867.5594426)

- Final Estimates

In [428]:
Bool_val = True
TotalDays = 31
ConsumedDays = 11

conditions = [(advance['Date_Check'] == Bool_val), (advance['BOL/BOC'] == "BOL") ]
choices = [ 0, advance['Line Item Net Budget'] ]

advance['Final Estimates'] = np.select( conditions, choices,
    default=np.minimum(
        ((advance['Net Spend'] * TotalDays) / ConsumedDays),
        advance['Line Item Net Budget']
    )
)


In [429]:
advance['Final Estimates'].sum()

np.float64(1005913681.2733545)

- Private brand


In [430]:
advance['LookupKey'] = advance['New Supercat'].astype(str) + advance['Brand'].astype(str) + advance['New Alpha/MP'].astype(str)
Tagging['LookupKey'] = Tagging['Supercat_PB'].astype(str) + Tagging['Brand_PB'].astype(str) + Tagging['Type_PB'].astype(str)

tagging_map = Tagging.drop_duplicates('LookupKey').set_index('LookupKey')['PrivateBrand']

advance['Private Brand'] = advance['LookupKey'].map(tagging_map).fillna(0)


In [399]:
# advance['Private Brand'].value_counts()

- KAM/NKAM


In [432]:
map_BUxBrand = kam_nkam.drop_duplicates('BUxBrand').set_index('BUxBrand')['Owner']
map_Brand = kam_nkam.drop_duplicates('Brand-1').set_index('Brand-1')['Owner']
advance['BUxBrand'] = advance['New BU'].astype(str) + advance['Brand'].astype(str)

condition = (advance['New Supercat'] == "Automobile")

BUxBrand_mapped = advance['BUxBrand'].map(map_BUxBrand).fillna("KAM")

final_map = advance['Brand'].map(map_Brand).fillna(BUxBrand_mapped)

conditions = [condition]
choices = ["KAM"]

advance['KAM/NKAM'] = np.select(conditions, choices, default=final_map)


In [438]:
advance[advance['KAM/NKAM'] == "KAM"]['Final Estimates'].sum()

np.float64(994849444.5197182)

- BMP/UMP

In [444]:
advance['BMP/UMP'] = np.where(advance['New Supercat'] == "Automobile", "BMP",np.where(advance['Alpha_MP'] == "BMP", "BMP","NonBMP"))

In [446]:
# advance['BMP/UMP'].value_counts()

- Internal RO Number

In [447]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['Internal RO number']

advance['Internal RO NO'] = advance['Line Item Id'].map(billing_map).fillna(0)


- External RO NO

In [448]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['External RO number']

advance['External RO NO'] = advance['Line Item Id'].map(billing_map).fillna(0)

- Advertiser

In [449]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['Advertiser']

advance['Advertiser'] = advance['Line Item Id'].map(billing_map).fillna(0)

- PAN Number

In [450]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['Pan Card Number']

advance['PAN Number'] = advance['Line Item Id'].map(billing_map).fillna(0)

- GSTIN

In [451]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['GST']

advance['GSTIN'] = advance['Line Item Id'].map(billing_map).fillna(0)

- RO Link

In [452]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['RO Link']

advance['RO Link'] = advance['Line Item Id'].map(billing_map).fillna(0)

- Internal Additional Document

In [453]:
billing_map = billing.drop_duplicates('Line Item Id').set_index('Line Item Id')['Internal Additional Document']

advance['Additional Document'] = advance['Line Item Id'].map(billing_map).fillna(0)

- loaded vs Estimates

In [454]:
advance['Loaded Vs Estimates'] = advance['Line Item Net Budget'] - advance['Final Estimates']

In [455]:
advance['Loaded Vs Estimates'].sum()

np.float64(208142531.45664546)

- Net Spend Vs Estimates

In [456]:
advance['NetSpend vs Estimates'] = advance['Line Item Net Budget'] - advance['Net Spend']

In [458]:
advance['NetSpend vs Estimates'].sum()

np.float64(880682135.1439)